### This notebook is about learning how to use Dataset and Dataloader inside our NN pipeline

In [6]:
import pandas as pd
import numpy as np
import torch
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import Dataset, DataLoader

### ***Using Brest Cancer data with Dataset and DataLoader***
**Use of Dataset and DataLoader->**
- Dataset is used to load the data from the files or server into the system where we will run the model and evaluate.
    - It is used to create a complete pipeline where we can load, and preprocess the entire data all in one go.

- DataLoader is used to load the data into batches instead of using the entire data all at once for training the model.
    - It can also shuffle the data inside the pipeline itself, to create unbiased training data


In [29]:
df = pd.read_csv("breast_cancer.csv")
X = df.drop(['id','diagnosis','Unnamed: 32'],axis=1)
y = df['diagnosis']

In [30]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2, random_state=42)

In [31]:
# Standardize the data and LabelEncode the Output
scaler = StandardScaler()
encode = LabelEncoder()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
y_train_enc = encode.fit_transform(y_train)
y_test_enc = encode.transform(y_test)

In [32]:
# Convert these into tensors of dtype float32
X_train_scaled = torch.from_numpy(X_train_scaled).to(dtype=torch.float32)
X_test_scaled = torch.from_numpy(X_test_scaled).to(dtype=torch.float32)
y_train_enc = torch.from_numpy(y_train_enc).to(dtype=torch.float32).view(-1,1)
t_test_enc = torch.from_numpy(y_test_enc).to(dtype=torch.float32).view(-1,1)

In [33]:
## Create dataset and dataloader to get the data
class CustomDataset(Dataset):
    def __init__(self, x_features , y_label):
        self.features = x_features
        self.labels = y_label

    def __len__(self):
        return len(self.features)

    def __getitem__(self , idx):
        return self.features[idx], self.labels[idx]

In [34]:
# forming the train and test datasets
train_dataset = CustomDataset(X_train_scaled , y_train_enc)
test_dataset = CustomDataset(X_test_scaled , y_test_enc) 

In [35]:
# forming the train and test dataloaders
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle=True)
test_loader = DataLoader(test_dataset , batch_size = 32 , shuffle=True)

In [41]:
# Creating the model
import torch.nn as nn
class MySimpleNN(nn.Module):

    def __init__(self , num_features):

        super().__init__()
        self.network = nn.Sequential( # building a simple neural net with only 1 layer
            nn.Linear(num_features , 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        out = self.network(features)
        return out

In [42]:
# create parameters
learning_rate = 0.1
epochs = 100

#create model
model = MySimpleNN(X_train_scaled.shape[1])

# create optimizer
optimizer = torch.optim.SGD(model.parameters() , lr = learning_rate)

# define the loss function
loss_function = nn.BCELoss()


In [ ]:
## Now let us run the model and train it!!

for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        # forward pass
        y_pred = model(batch_features)

        # loss calculate
        loss = loss_function(y_pred , batch_labels)

        # clear gradients before hand
        optimizer.zero_grad()

        # backpropagation
        loss.backward()

        # update the weights and bias using optimizer

        optimizer.step()

        total_epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_epoch_loss/len(train_loader):.4f}") 



Epoch 1/100 | Loss: 0.3324
Epoch 2/100 | Loss: 0.1950
Epoch 3/100 | Loss: 0.1506
Epoch 4/100 | Loss: 0.1293
Epoch 5/100 | Loss: 0.1258
Epoch 6/100 | Loss: 0.1104
Epoch 7/100 | Loss: 0.1044
Epoch 8/100 | Loss: 0.1031
Epoch 9/100 | Loss: 0.0958
Epoch 10/100 | Loss: 0.0925
Epoch 11/100 | Loss: 0.0905
Epoch 12/100 | Loss: 0.0889
Epoch 13/100 | Loss: 0.0893
Epoch 14/100 | Loss: 0.0892
Epoch 15/100 | Loss: 0.1153
Epoch 16/100 | Loss: 0.0834
Epoch 17/100 | Loss: 0.0797
Epoch 18/100 | Loss: 0.0833
Epoch 19/100 | Loss: 0.0789
Epoch 20/100 | Loss: 0.0850
Epoch 21/100 | Loss: 0.0792
Epoch 22/100 | Loss: 0.0786
Epoch 23/100 | Loss: 0.0734
Epoch 24/100 | Loss: 0.0773
Epoch 25/100 | Loss: 0.0722
Epoch 26/100 | Loss: 0.0768
Epoch 27/100 | Loss: 0.0750
Epoch 28/100 | Loss: 0.0700
Epoch 29/100 | Loss: 0.0702
Epoch 30/100 | Loss: 0.0692
Epoch 31/100 | Loss: 0.0699
Epoch 32/100 | Loss: 0.0681
Epoch 33/100 | Loss: 0.0676
Epoch 34/100 | Loss: 0.0670
Epoch 35/100 | Loss: 0.0671
Epoch 36/100 | Loss: 0.0662
E

In [44]:
# Evaluation

model.eval() # this sets the model to evaluation mode
accuracy = []
with torch.no_grad():
    for batch_features , batch_labels in test_loader:
        # forward pass
        y_pred = model(batch_features)

        # convert probs to binary
        y_pred = (y_pred>0.8).float() 

        # check the accuracy and append it
        accuracy.append(accuracy_score(y_pred , batch_labels))

# final accuracy score

print(f' Final Accuracy Score -> {round(sum(accuracy)/len(accuracy),3)}')



 Final Accuracy Score -> 0.984
